# Federated Learning to Hybrid Verification Tutorial

This notebook demonstrates a full Python SDK v2 flow:

1. Simulate local training on multiple nodes
2. Aggregate updates and compress gradients
3. Verify zk-style proof payloads
4. Run hybrid SNARK/STARK verification using high-level wrappers
5. Render a human-readable operator summary

In [ ]:
from pathlib import Path
import json
import random

from mohawk import (
    MohawkNode,
    HybridProofCheck,
)

random.seed(7)

In [ ]:
def simulate_local_training(node_id: str, dims: int = 8):
    gradient = [round(random.uniform(-0.4, 0.4), 5) for _ in range(dims)]
    loss = round(random.uniform(0.08, 0.5), 5)
    samples = random.randint(80, 180)
    return {
        "node_id": node_id,
        "gradient": gradient,
        "weight": samples / 100.0,
        "loss": loss,
        "samples": samples,
    }

updates = [simulate_local_training(f"edge-{i:03d}") for i in range(1, 6)]
updates

In [ ]:
node = MohawkNode()
node.start(config_path="capabilities.json", node_id="tutorial-aggregator")

In [ ]:
aggregation_result = node.aggregate(updates)
stream_result = node.stream_aggregate(
    [u["gradient"] for u in updates],
    format="int8",
    max_norm=1.0,
)

{
    "aggregation_message": aggregation_result.get("message"),
    "compressed_bytes": stream_result.get("compressed_bytes"),
    "compression_ratio": stream_result.get("compression_ratio"),
}

In [ ]:
proof_payload = {
    "proof": "0x" + "ab" * 80,
    "public_inputs": ["round=1", "nodes=5"],
    "scheme": "groth16",
}

proof_result = node.verify_proof(proof_payload)
proof_result

In [ ]:
hybrid_check = HybridProofCheck(
    snark_proof="s" * 128,
    stark_proof="t" * 64,
    mode="both",
    stark_backend="simulated_fri",
)

hybrid_receipt = node.verify_hybrid(hybrid_check)
hybrid_receipt

In [ ]:
human_summary = {
    "training_nodes": len(updates),
    "avg_loss": round(sum(u["loss"] for u in updates) / len(updates), 5),
    "proof_verified": bool(proof_result.get("success", False)),
    "hybrid_verified": hybrid_receipt.success,
    "hybrid_mode": hybrid_receipt.mode or "unknown",
    "next_action": (
        "Proceed with model publication and bridge settlement"
        if hybrid_receipt.success
        else "Hold publish and investigate verifier logs"
    ),
}

summary_path = Path("results/metrics/fl_tutorial_human_summary.json")
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(human_summary, indent=2), encoding="utf-8")
human_summary